In [ ]:
from Import_loader import *

In [ ]:
import torch

In [ ]:
# Experiment
utils.load_config('sphere_pos.json')
config['ref_position'] = 1

# Setup
params = setup_utils.setup_experiment(config)
scene = params['scene']
ref = params['ref_msr']
update_fn = params['update_fn']
gt = setup_utils.get_gt()

print(gt)

In [ ]:
hyper_params = {
    'nsamples': 1,
    'sigma': 0.0125,
    'initial_translation': [0.0],
    'gt_translation': [gt],
    'learning_rate': 2e-2,
    'epochs': 50,
    'sigma_annealing': False,
    'anneal_const_first': 20,
    'anneal_const_last': 0,
    'anneal_sigma_min': 0.001,
}

ctx_args = {
    'scene': scene, 'init_pos': [0.0], 'gt_msr': ref,
    'update_fn': update_fn, 'loss_fn': losses.torch_fft_l2,
    'sampler': 'importance', 'antithetic': True, 'nsamples': hyper_params['nsamples'],
    'sigma': hyper_params['sigma'], 'device': config['device']
}

In [ ]:
# Set up initial and gt translation
initial_translation = torch.tensor(hyper_params['initial_translation'], requires_grad=True, device=config['device'])
gt_translation = torch.tensor(hyper_params['gt_translation'], device=config['device'])

# Set up optimizer:
optim = torch.optim.Adam([initial_translation], lr=hyper_params['learning_rate'])

In [ ]:
theta_optim = opt.run_optimization(hparams=hyper_params,
                               optim=optim,
                               theta=initial_translation,
                               gt_theta=gt_translation,
                               ctx_args=ctx_args,
                               schedule_fn=opt.run_scheduler_step,
                               update_fn=update_fn)

In [ ]:
print(theta_optim)